# Waste Classifier with PyTorch: organic vs inorganic

This notebook trains an image classifier that looks at a photo (or a webcam frame) and tells you if it shows organic trash, inorganic trash, or nothing at all.

# 1. Install the libraries

Colab already has PyTorch, so this only installs stuff on your laptop. If everything's already there, pip just skips it.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if not IN_COLAB:
    %pip install -q torch==2.14.1 torchvision==0.29.1 opencv-python==5.0.0.93 matplotlib==3.11.2 scikit-learn==1.9.1 pillow==12.3.0

# 2. Imports and device

`DEVICE` picks the fastest hardware you have: an NVIDIA GPU (`cuda`), an Apple Silicon GPU (`mps`), or just the CPU.
Training on CPU works, it's just slower.

In [ ]:
import random
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("PyTorch", torch.__version__, "| running on", DEVICE)

# 3. Configuration

All the knobs live in one place. Change things here instead of hunting through the code.

In [ ]:
CONFIGURATION = {
    "DATA_DIR": "data",                              # one subfolder per class
    "CLASS_NAMES": ["organik", "anorganik", "kosong"],
    "BACKGROUND_CLASS": "kosong",                    # "nothing here" class -> no output
    "MIN_IMAGES_PER_CLASS": 10,                      # classes with fewer photos are skipped
    "IM_SIZE": 224,                                  # images get resized to 224x224
    "BATCH_SIZE": 32,                                # lower it (e.g. 16) if you run out of memory
    "LEARNING_RATE": 1e-3,
    "N_EPOCHS": 15,
    "PATIENCE": 5,                                   # stop early after this many epochs without improvement
    "DROPOUT_RATE": 0.2,
    "SPLIT": (0.7, 0.2, 0.1),                        # train / validation / test
    "SEED": 42,                                      # same seed = same split every time
    "PRETRAINED": True,                              # start from ImageNet weights (keep this on)
    "CONF_THRESHOLD": 0.6,                           # min confidence to say "there's trash"
    "MODEL_PATH": "models/pytorch_sampah.pt",
}

random.seed(CONFIGURATION["SEED"])
np.random.seed(CONFIGURATION["SEED"])
torch.manual_seed(CONFIGURATION["SEED"])

# On Colab, read photos from and save the model to Google Drive so nothing gets lost.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_FOLDER = "/content/drive/MyDrive/klasifikasi-sampah-notebook"   # change if you put it elsewhere
    CONFIGURATION["DATA_DIR"] = f"{DRIVE_FOLDER}/data"
    CONFIGURATION["MODEL_PATH"] = f"{DRIVE_FOLDER}/models/pytorch_sampah.pt"

# 4. Check your photos

Let's count what's in each class folder and catch any broken files before they crash training.
A class needs at least `MIN_IMAGES_PER_CLASS` photos, otherwise it gets skipped for now.

If this cell complains that there aren't enough classes, go drop some photos into `data/` and run it again.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
data_dir = Path(CONFIGURATION["DATA_DIR"])


def list_photos(folder):
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def is_valid_image(path):
    try:
        with Image.open(path) as image:
            image.verify()
        return True
    except Exception:
        return False


photos_per_class = {}
for name in CONFIGURATION["CLASS_NAMES"]:
    folder = data_dir / name
    folder.mkdir(parents=True, exist_ok=True)  # creates the empty folders on the first run
    photos = list_photos(folder)
    good = [p for p in photos if is_valid_image(p)]
    photos_per_class[name] = good
    note = f"  ({len(photos) - len(good)} broken file(s) skipped)" if len(good) < len(photos) else ""
    print(f"{name:<10} {len(good):>5} photos{note}")

CLASS_NAMES = [n for n, p in photos_per_class.items() if len(p) >= CONFIGURATION["MIN_IMAGES_PER_CLASS"]]
skipped = [n for n in CONFIGURATION["CLASS_NAMES"] if n not in CLASS_NAMES]
if skipped:
    print(f"\nSkipping {skipped}: fewer than {CONFIGURATION['MIN_IMAGES_PER_CLASS']} photos.")

assert len(CLASS_NAMES) >= 2, f"Need at least 2 classes with photos. Add photos to {data_dir.resolve()} and run this cell again."
NUM_CLASSES = len(CLASS_NAMES)
print("\nClasses used:", CLASS_NAMES)

# 5. Split into train, validation and test

- **train**: what the model learns from
- **validation**: checked after every epoch to see if it's actually getting better
- **test**: kept aside until the very end, for an honest final score

In [ ]:
paths, labels = [], []
for index, name in enumerate(CLASS_NAMES):
    paths += photos_per_class[name]
    labels += [index] * len(photos_per_class[name])

train_frac, val_frac, test_frac = CONFIGURATION["SPLIT"]
seed = CONFIGURATION["SEED"]

# First cut: train vs the rest. Second cut: the rest -> val and test.
# stratify=... keeps the class balance the same in every split.
train_paths, rest_paths, train_labels, rest_labels = train_test_split(
    paths, labels, train_size=train_frac, stratify=labels, random_state=seed)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    rest_paths, rest_labels, train_size=val_frac / (val_frac + test_frac), stratify=rest_labels, random_state=seed)

print(f"{'class':<10}{'train':>8}{'val':>8}{'test':>8}")
for index, name in enumerate(CLASS_NAMES):
    counts = [split.count(index) for split in (train_labels, val_labels, test_labels)]
    print(f"{name:<10}" + "".join(f"{c:>8}" for c in counts))

# 6. Datasets, augmentation and data loaders

**Augmentation** means randomly cropping, flipping, rotating and changing the brightness of training photos every time they're loaded.
The model never sees the exact same picture twice, so it learns "what a banana peel looks like" instead of memorizing your photos.
Validation and test photos are only resized, with no tricks, so the scores stay fair.

`WasteDataset` is a tiny custom PyTorch dataset: give it a list of file paths and labels, and it hands back ready-to-use tensors.
`DataLoader` then groups them into batches.

In [ ]:
IM_SIZE = CONFIGURATION["IM_SIZE"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]   # EfficientNet was trained with these, so we normalize the same way
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IM_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IM_SIZE, IM_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class WasteDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = Image.open(self.paths[index])
        image = ImageOps.exif_transpose(image).convert("RGB")  # fixes sideways phone photos
        return self.transform(image), self.labels[index]


train_dataset = WasteDataset(train_paths, train_labels, train_transform)
val_dataset = WasteDataset(val_paths, val_labels, eval_transform)
test_dataset = WasteDataset(test_paths, test_labels, eval_transform)

# num_workers=0 keeps things simple and avoids multiprocessing issues in notebooks on Windows.
BATCH_SIZE = CONFIGURATION["BATCH_SIZE"]
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"batches -> train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

# 7. Take a look at the training data

Always eyeball your data before training. You'll catch wrong labels, weird photos, or augmentation that's way too aggressive.

In [ ]:
def to_displayable(tensor):
    """Undo the normalization so the image looks normal again."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


images, labels = next(iter(train_loader))
plt.figure(figsize=(12, 8))
for i in range(min(12, len(images))):
    plt.subplot(3, 4, i + 1)
    plt.imshow(to_displayable(images[i]))
    plt.title(CLASS_NAMES[labels[i]])
    plt.axis("off")
plt.suptitle("Training photos (with augmentation)")
plt.tight_layout()
plt.show()

# 8. Build the model

1. Load EfficientNet-B0 with its ImageNet weights.
2. **Freeze** the feature extractor (the part that already knows edges, textures and shapes), so it doesn't change.
3. Swap the last layer for a new one with exactly as many outputs as we have classes.

Only that new last layer gets trained, which is fast and needs little data.

In [ ]:
def build_model(num_classes, pretrained):
    weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b0(weights=weights)
    if pretrained:
        for param in model.features.parameters():
            param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(CONFIGURATION["DROPOUT_RATE"]),
        nn.Linear(in_features, num_classes),
    )
    return model


model = build_model(NUM_CLASSES, CONFIGURATION["PRETRAINED"]).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total:,} | trainable: {trainable:,}")

# 9. Loss, optimizer and learning-rate scheduler

- **Loss** (`CrossEntropyLoss`): one number that says how wrong the predictions are. Training tries to push it down.
- **Optimizer** (`Adam`): the thing that actually nudges the weights to lower the loss.
- **Scheduler** (`ReduceLROnPlateau`): if validation loss stops improving, it halves the learning rate so the model can fine-tune with smaller steps.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIGURATION["LEARNING_RATE"],
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

# 10. The training loop

In PyTorch you write the loop yourself. Each step does the same 4 things:

1. **forward**: run the images through the model
2. **loss**: compare predictions with the real labels
3. **backward**: work out how each weight should change
4. **step**: actually change the weights

`run_epoch` does one full pass over a data loader. With `train=True` it learns, with `train=False` it only measures.

In [ ]:
def run_epoch(model, loader, train):
    model.train(train)
    if train and CONFIGURATION["PRETRAINED"]:
        model.features.eval()  # keep the frozen backbone's BatchNorm stats as they are

    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            count += images.size(0)

    return total_loss / count, correct / count

# 11. Train!

Every epoch, it trains on the training set and then checks the validation set.

- Whenever validation loss hits a new low, the model gets saved to `MODEL_PATH`, so we always keep the best version.
- If it doesn't improve for `PATIENCE` epochs in a row, training stops early (**early stopping**). No point burning time.

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
epochs_without_improvement = 0
model_path = Path(CONFIGURATION["MODEL_PATH"])
model_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, CONFIGURATION["N_EPOCHS"] + 1):
    start = time.time()
    train_loss, train_acc = run_epoch(model, train_loader, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, train=False)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    print(f"Epoch {epoch:>2}/{CONFIGURATION['N_EPOCHS']} | "
          f"train loss {train_loss:.4f}, acc {train_acc:.3f} | "
          f"val loss {val_loss:.4f}, acc {val_acc:.3f} | {time.time() - start:.0f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save({"state_dict": model.state_dict(), "class_names": CLASS_NAMES}, model_path)
        print("   -> new best model saved")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CONFIGURATION["PATIENCE"]:
            print(f"No improvement for {CONFIGURATION['PATIENCE']} epochs, stopping early.")
            break

# 12. Training curves

How to read them:

- both losses go down together → healthy 👍
- train loss keeps dropping but validation loss goes **up** → **overfitting**: the model is memorizing. Get more varied photos.
- neither goes down → something's off with the data or the labels.

In [ ]:
epoch_numbers = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epoch_numbers, history["train_loss"], label="train")
plt.plot(epoch_numbers, history["val_loss"], label="validation")
plt.title("Loss")
plt.xlabel("epoch")
plt.xticks(epoch_numbers)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epoch_numbers, history["train_acc"], label="train")
plt.plot(epoch_numbers, history["val_acc"], label="validation")
plt.title("Accuracy")
plt.xlabel("epoch")
plt.xticks(epoch_numbers)
plt.legend()

plt.tight_layout()
plt.show()

# 13. Evaluate on the test set

Now for the honest score, on photos the model has never seen. We load the best saved model first.

- **precision**: when it says "organik", how often is it right?
- **recall**: out of all the real organic photos, how many did it catch?
- **f1-score**: precision and recall rolled into one number
- **confusion matrix**: rows are the real class, columns are what the model guessed. Everything off the diagonal is a mistake.

In [ ]:
checkpoint = torch.load(model_path, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(DEVICE))
        y_pred += outputs.argmax(dim=1).cpu().tolist()
        y_true += labels.tolist()

class_ids = list(range(NUM_CLASSES))
print(classification_report(y_true, y_pred, labels=class_ids, target_names=CLASS_NAMES, digits=3, zero_division=0))

ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=class_ids), display_labels=CLASS_NAMES).plot(cmap="Blues")
plt.title("Confusion matrix (test set)")
plt.show()

# 14. See the predictions

A quick visual check on some test photos. Green title = correct, red = wrong.
Looking at the red ones usually tells you what kind of photos to add next.

In [ ]:
sample = random.sample(range(len(test_dataset)), k=min(12, len(test_dataset)))
plt.figure(figsize=(12, 9))
with torch.no_grad():
    for plot_index, data_index in enumerate(sample):
        image, label = test_dataset[data_index]
        probs = torch.softmax(model(image.unsqueeze(0).to(DEVICE)), dim=1)[0]
        pred = int(probs.argmax())
        plt.subplot(3, 4, plot_index + 1)
        plt.imshow(to_displayable(image))
        plt.title(f"real: {CLASS_NAMES[label]}\npred: {CLASS_NAMES[pred]} ({probs[pred]:.2f})",
                  color="green" if pred == label else "red")
        plt.axis("off")
plt.tight_layout()
plt.show()

# 15. Turn probabilities into the final output

The model always gives a probability for every class. This little rule turns that into the output we actually want:

- top guess is `kosong` → **no output**
- the model isn't confident enough (below `CONF_THRESHOLD`) → **no output**
- otherwise → print `organik` or `anorganik`

In [ ]:
def decide(probs, class_names):
    """probs -> (output, top_label, top_conf). output is None when there's no trash."""
    index = int(np.argmax(probs))
    label, conf = class_names[index], float(probs[index])
    if label == CONFIGURATION["BACKGROUND_CLASS"] or conf < CONFIGURATION["CONF_THRESHOLD"]:
        return None, label, conf
    return label, label, conf


def predict_frame(model, frame_bgr, class_names):
    """Classify one OpenCV frame (BGR) and apply the output rule."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    tensor = eval_transform(Image.fromarray(rgb)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
    return decide(probs, class_names)

# 16. Try it on a single photo

This reads a photo with OpenCV (`cv2`), exactly like the webcam will. If nothing gets printed, the model thinks there's no trash.
Change `TEST_IMAGE` to any photo you want to try.

In [ ]:
TEST_IMAGE = str(test_paths[0])   # e.g. "my_photo.jpg"

frame = cv2.imread(TEST_IMAGE)
output, top_label, top_conf = predict_frame(model, frame, CLASS_NAMES)
if output:
    print(output)

plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f"top guess: {top_label} ({top_conf:.2f})")
plt.axis("off")
plt.show()

# 17. Load the saved model later

The best model is already saved at `MODEL_PATH` (the class names are saved inside it too).
Here's how to load it back in a fresh session without retraining. If you trained on Colab, download that file to your laptop's `models/` folder first.

In [ ]:
def load_model(path):
    checkpoint = torch.load(path, map_location=DEVICE)
    class_names = checkpoint["class_names"]
    loaded = build_model(len(class_names), pretrained=False)  # architecture only; weights come from the file
    loaded.load_state_dict(checkpoint["state_dict"])
    return loaded.to(DEVICE).eval(), class_names


model, CLASS_NAMES = load_model(CONFIGURATION["MODEL_PATH"])
print("Loaded model with classes:", CLASS_NAMES)

# 18. Live webcam

This is the real deal: it reads frames from your webcam, classifies each one, and prints the result **only when it changes**,
so your screen doesn't get flooded with the same word 30 times a second.

- It only works on your laptop. Colab can't reach your webcam.
- Flip `RUN_WEBCAM` to `True` to start it. It's off by default, so *Run all* doesn't get stuck here.
- Press **Q** in the video window to stop.

In [ ]:
RUN_WEBCAM = False   # set to True on your laptop

if RUN_WEBCAM and not IN_COLAB:
    cap = cv2.VideoCapture(0)   # 0 = default webcam, try 1 if you have another one
    last_output = None
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        output, top_label, top_conf = predict_frame(model, frame, CLASS_NAMES)

        if output != last_output:   # only print when the result changes
            if output:
                print(output)
            last_output = output

        text = f"{output} {top_conf:.2f}" if output else f"no trash ({top_label} {top_conf:.2f})"
        cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.imshow("Waste classifier - press Q to quit", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
else:
    print("Webcam is off. Set RUN_WEBCAM = True (laptop only) to start it.")